# Pipeline médaillon — empreinte carbone last-mile (v2)

Bronze → Silver → Gold sur le dataset Amazon Last Mile Routing Research Challenge (`model_build_inputs`, vrai volume) + facteur DEFRA 2026.

**Changements vs v1** : audit consolidé (toutes les colonnes, doublons inclus, pas seulement `zone_id`), chargement du vrai jeu d'entraînement avec streaming S3 (pas de téléchargement du fichier de 1,8 Go), vraie séquence de livraison (`actual_sequences`), EDA complète, comparaison diesel/électrique.

## Imports et chemin du projet

In [1]:
import json
import math
import io
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import ijson
import boto3
from botocore import UNSIGNED
from botocore.config import Config

data_dir = Path(r'C:\Users\ASUS\Documents\projet_multi_agents\data')
bronze_dir = data_dir / "bronze"
silver_dir = data_dir / "silver"
gold_dir = data_dir / "gold"
for d in (bronze_dir, silver_dir, gold_dir):
    d.mkdir(exist_ok=True)

s3 = boto3.client(
    's3',
    config=Config(signature_version=UNSIGNED, read_timeout=600, retries={'max_attempts': 5})
)
BUCKET = 'amazon-last-mile-challenges'
BUILD_PREFIX = 'almrrc2021/almrrc2021-data-training/model_build_inputs/'

print("data_dir :", data_dir)

data_dir : C:\Users\ASUS\Documents\projet_multi_agents\data


## 1. Bronze — chargement du vrai jeu d'entraînement

`route_data.json` (79 Mo) et `actual_sequences.json` (9,7 Mo) sont téléchargés en entier. `travel_times.json` (1,8 Go) est **streamé directement depuis S3** avec `ijson`, sans jamais être écrit sur disque — on ne garde en mémoire que les routes sélectionnées.

In [3]:
import io
import sys
import json
from boto3.s3.transfer import TransferConfig


def load_light_json_from_s3(key: str, cache_name: str) -> dict:
    """Téléchargés une fois avec reprise automatique en cas de coupure,
    puis mis en cache localement (réutilisé si déjà présent)."""
    dest = bronze_dir / "model_build_inputs" / cache_name
    dest.parent.mkdir(exist_ok=True, parents=True)
    if not dest.exists():
        s3.download_file(BUCKET, key, str(dest))
    return json.load(open(dest))


def download_travel_times(force=False):
    """Fichier lourd (1,8 Go) : téléchargé une seule fois, avec reprise
    automatique par morceau (multipart) gérée nativement par boto3.
    Chunks de 25 Mo."""
    dest = bronze_dir / "model_build_inputs" / "travel_times.json"
    dest.parent.mkdir(exist_ok=True, parents=True)
    if dest.exists() and not force:
        print(f"Déjà présent : {dest} ({dest.stat().st_size / 1e9:.2f} Go)")
        return dest

    transfer_config = TransferConfig(
        multipart_threshold=1024 * 1024 * 25, max_concurrency=10,
        multipart_chunksize=1024 * 1024 * 25, use_threads=True,
    )

    def progress(bytes_transferred, total=[0]):
        total[0] += bytes_transferred
        pct = total[0] / (1.8 * 1024**3) * 100
        sys.stdout.write(f"\r  ... {total[0] / 1e6:.0f} Mo téléchargés (~{pct:.1f}%)")
        sys.stdout.flush()

    s3.download_file(
        BUCKET, BUILD_PREFIX + "travel_times.json", str(dest),
        Config=transfer_config, Callback=progress,
    )
    print(f"\nTerminé : {dest}")
    return dest


def stream_travel_times_local(path, n_routes=1000):
    """Parsing du fichier lourd EN LOCAL (aucun réseau impliqué,
    donc aucun risque de timeout). n_routes borne la RAM utilisée."""
    travel_times = {}
    with open(path, "rb") as f:
        buffered = io.BufferedReader(f, buffer_size=1024 * 1024)
        for i, (route_id, tt) in enumerate(ijson.kvitems(buffered, '')):
            travel_times[route_id] = tt
            if (i + 1) % 200 == 0:
                print(f"  ... {i + 1} routes parsées")
            if n_routes and len(travel_times) >= n_routes:
                break
    return travel_times


def load_build_bronze(n_routes=1000):
    routes_full = load_light_json_from_s3(BUILD_PREFIX + "route_data.json", "route_data.json")
    sequences_full = load_light_json_from_s3(BUILD_PREFIX + "actual_sequences.json", "actual_sequences.json")

    tt_path = download_travel_times()
    travel_times = stream_travel_times_local(tt_path, n_routes=n_routes)
    selected_set = set(travel_times.keys())

    routes = {rid: routes_full[rid] for rid in selected_set if rid in routes_full}
    sequences = {rid: sequences_full[rid] for rid in selected_set if rid in sequences_full}

    print(f"\nRoutes : {len(routes)} | séquences réelles : {len(sequences)} | travel_times : {len(travel_times)}")
    return routes, travel_times, sequences


routes, travel_times, sequences = load_build_bronze(n_routes=1000)

  ... 1766 Mo téléchargés (~91.4%)

RetriesExceededError: Max Retries Exceeded

## 2. Audit consolidé — toutes les colonnes, en un seul bloc

Valeurs manquantes sur l'ensemble des champs (route ET arrêt), doublons (route_id, stop_id dupliqué dans une même route), plages de valeurs incohérentes (lat/lng hors bornes, capacité négative ou nulle, route sans arrêt).

In [ ]:
def audit_full(routes: dict) -> dict:
    route_rows, stop_rows = [], []
    for route_id, route in routes.items():
        route_rows.append({
            "route_id": route_id,
            "station_code": route.get("station_code"),
            "date": route.get("date_YYYY_MM_DD"),
            "departure_time": route.get("departure_time_utc"),
            "capacity_cm3": route.get("executor_capacity_cm3"),
            "nb_stops": len(route.get("stops", {})),
        })
        for stop_id, stop in route.get("stops", {}).items():
            stop_rows.append({
                "route_id": route_id,
                "stop_id": stop_id,
                "lat": stop.get("lat"),
                "lng": stop.get("lng"),
                "type": stop.get("type"),
                "zone_id": stop.get("zone_id"),
            })

    route_df = pd.DataFrame(route_rows)
    stop_df = pd.DataFrame(stop_rows)

    print("=== Valeurs manquantes -- niveau route ===")
    print((route_df.isna().mean() * 100).round(2).astype(str) + " %")
    print("\n=== Valeurs manquantes -- niveau arrêt ===")
    print((stop_df.isna().mean() * 100).round(2).astype(str) + " %")

    print("\n=== Doublons ===")
    print("route_id dupliqué :", route_df["route_id"].duplicated().sum())
    dup_stops = stop_df.duplicated(subset=["route_id", "stop_id"]).sum()
    print("stop_id dupliqué au sein d'une même route :", dup_stops)

    print("\n=== Cohérence des plages de valeurs ===")
    print("lat hors [-90, 90] :", (~stop_df["lat"].between(-90, 90)).sum())
    print("lng hors [-180, 180] :", (~stop_df["lng"].between(-180, 180)).sum())
    print("capacité <= 0 :", (route_df["capacity_cm3"] <= 0).sum())
    print("routes sans arrêt :", (route_df["nb_stops"] == 0).sum())

    return {"route_df": route_df, "stop_df": stop_df}

audit_result = audit_full(routes)
route_raw, stops_raw = audit_result["route_df"], audit_result["stop_df"]

## 3. Silver — nettoyage

`zone_id` (catégoriel) : imputation par le mode du stop géographiquement le plus proche dans la même route, repli sur le mode global sinon. Reconstruction des segments à partir de la **vraie séquence** (`actual_sequences`), distance haversine, filtre qualité (vitesse implicite, contrôle et non méthode de calcul).

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2) -> float:
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlambda / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


def impute_zone_id(stops_df: pd.DataFrame) -> pd.DataFrame:
    df = stops_df.copy()
    print(f"Taux de zone_id manquant : {df['zone_id'].isna().mean():.2%}")
    global_mode = df["zone_id"].mode(dropna=True)
    global_mode = global_mode.iloc[0] if not global_mode.empty else "unknown"

    for route_id, group in df[df["zone_id"].isna()].groupby("route_id"):
        known = df[(df["route_id"] == route_id) & df["zone_id"].notna()]
        for idx, row in group.iterrows():
            if known.empty:
                df.loc[idx, "zone_id"] = global_mode
                continue
            dists = known.apply(lambda r: haversine_km(row["lat"], row["lng"], r["lat"], r["lng"]), axis=1)
            df.loc[idx, "zone_id"] = known.loc[dists.idxmin(), "zone_id"]
    return df

stops_clean = impute_zone_id(stops_raw)


def build_legs(routes: dict, travel_times: dict, sequences: dict) -> pd.DataFrame:
    rows = []
    for route_id, route in routes.items():
        seq_entry = sequences.get(route_id, {}).get("actual")
        stops = route["stops"]
        if not seq_entry:
            continue  # pas de vérité terrain pour cette route, on l'ignore ici
        ordered_stops = [seq_entry[str(i)] for i in range(len(seq_entry))]
        for i in range(len(ordered_stops) - 1):
            a, b = ordered_stops[i], ordered_stops[i + 1]
            if a not in stops or b not in stops:
                continue
            lat1, lng1 = stops[a]["lat"], stops[a]["lng"]
            lat2, lng2 = stops[b]["lat"], stops[b]["lng"]
            t = travel_times.get(route_id, {}).get(a, {}).get(b)
            rows.append({
                "route_id": route_id, "from_stop_id": a, "to_stop_id": b,
                "ordre_sequence": i, "temps_reel_sec": t,
                "distance_haversine_km": haversine_km(lat1, lng1, lat2, lng2),
            })
    return pd.DataFrame(rows)


def quality_filter(legs_df: pd.DataFrame, min_kmh=1, max_kmh=60) -> pd.DataFrame:
    df = legs_df.copy()
    df["vitesse_implicite_kmh"] = df["distance_haversine_km"] / (df["temps_reel_sec"] / 3600)
    df["segment_suspect"] = ~df["vitesse_implicite_kmh"].between(min_kmh, max_kmh)
    print(f"Segments suspects : {df['segment_suspect'].sum()} / {len(df)}")
    return df

legs_df = quality_filter(build_legs(routes, travel_times, sequences))
stops_clean.to_csv(silver_dir / "stops_clean.csv", index=False)
legs_df.to_csv(silver_dir / "legs_clean.csv", index=False)
print(f"\nSilver sauvegardé dans {silver_dir} -- fact potentiel : {len(legs_df)} lignes")

## 4. EDA — exploration réelle

Distributions, pas juste un résumé statistique.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(legs_df["distance_haversine_km"], bins=40)
axes[0, 0].set_title("Distribution des distances par segment (km)")

axes[0, 1].hist(legs_df["vitesse_implicite_kmh"].clip(0, 80), bins=40)
axes[0, 1].set_title("Distribution de la vitesse implicite (km/h)")

axes[1, 0].hist(route_raw["nb_stops"], bins=30)
axes[1, 0].set_title("Nombre d'arrêts par route")

axes[1, 1].hist(route_raw["capacity_cm3"] / 1e6, bins=30)
axes[1, 1].set_title("Capacité du van (m³)")

plt.tight_layout()
plt.savefig(silver_dir / "eda_distributions.png", dpi=120)
plt.show()

print("\nCorrélation distance / temps réel :", legs_df[["distance_haversine_km", "temps_reel_sec"]].corr().iloc[0, 1].round(3))
print("\nRépartition des zones (top 10) :")
print(stops_clean["zone_id"].value_counts().head(10))

## 5. DEFRA — facteurs d'émission (4 carburants Class III)

Facteur retenu pour le calcul réel : Class III Diesel. Les autres carburants disponibles pour cette classe (Petrol, Plug-in Hybrid, Battery Electric) sont chargés uniquement comme **scénarios de comparaison** ("et si la flotte changeait de motorisation ?"), pas pour le calcul principal.

In [ ]:
def load_defra_factors(xlsx_url: str) -> pd.DataFrame:
    factors = pd.read_excel(xlsx_url, sheet_name="Factors by Category", header=5)
    factors.columns = ["ID", "Scope", "Level1", "Level2", "Level3", "Level4", "ColumnText", "UOM", "GHGUnit", "Factor2026"]
    return factors[
        (factors["Level1"] == "Delivery vehicles") & (factors["Level2"] == "Vans")
        & (factors["Level3"] == "Class III (1.74 to 3.5 tonnes)")
        & (factors["GHGUnit"] == "kg CO2e") & (factors["UOM"] == "km")
    ][["ColumnText", "Factor2026"]].dropna()

defra_url = "https://assets.publishing.service.gov.uk/media/6a6c9748862aaf18d9c62ac9/ghg-conversion-factors-2026-flat-format-revised.xlsx"
defra_rows = load_defra_factors(defra_url)
print(defra_rows)  # doit afficher 4 lignes : Diesel, Petrol, Plug-in Hybrid Electric Vehicle, Battery Electric Vehicle

factors_by_fuel = dict(zip(defra_rows["ColumnText"], defra_rows["Factor2026"]))
factor_diesel = factors_by_fuel["Diesel"]
print(f"\nDiesel (retenu pour le calcul réel) : {factor_diesel} kg CO2e/km")
print("NB : le facteur BEV DEFRA ne compte que le tank-to-wheel (0 direct) -- comparaison illustrative uniquement, pas une vraie empreinte cycle de vie.")

## 6. Gold — dimensions et fait unique (star schema)

`dim_vehicle` contient les 4 carburants Class III (Diesel, Petrol, PHEV, BEV) pour permettre la comparaison. `fact_route_leg` calcule l'émission réelle (diesel) ET une colonne de scénario par carburant alternatif. Un seul fait, pas de duplication de grain.

In [ ]:
dim_route = route_raw.copy()
dim_stop = stops_clean[["stop_id", "lat", "lng", "zone_id", "type"]].drop_duplicates("stop_id")
dim_vehicle = pd.DataFrame([
    {"vehicle_class": f"van_class3_{fuel.lower().replace(' ', '_')}", "classe": "Class III", "carburant": fuel, "facteur_kg_co2e_km": val}
    for fuel, val in factors_by_fuel.items()
])

fact_route_leg = legs_df.copy()
fact_route_leg["vehicle_class"] = "van_class3_diesel"
fact_route_leg["emission_kg_co2e"] = fact_route_leg["distance_haversine_km"] * factor_diesel
# une colonne de scénario par carburant alternatif, pour comparaison -- toujours un seul fait, pas de duplication de grain
for fuel, val in factors_by_fuel.items():
    if fuel == "Diesel":
        continue
    col = f"emission_scenario_{fuel.lower().replace(' ', '_')}_kg_co2e"
    fact_route_leg[col] = fact_route_leg["distance_haversine_km"] * val

print("dim_route:", dim_route.shape, "| dim_stop:", dim_stop.shape, "| dim_vehicle:", dim_vehicle.shape, "| fact_route_leg:", fact_route_leg.shape)

scenario_cols = [c for c in fact_route_leg.columns if c.startswith("emission_scenario_")]
agg_dict = {"distance_totale_km": ("distance_haversine_km", "sum"), "emission_totale_kg_co2e": ("emission_kg_co2e", "sum"), "nb_segments": ("ordre_sequence", "count")}
for c in scenario_cols:
    agg_dict[c.replace("emission_", "emission_totale_")] = (c, "sum")
totals = fact_route_leg.groupby("route_id").agg(**agg_dict).reset_index()
for c in scenario_cols:
    total_col = c.replace("emission_", "emission_totale_")
    totals[total_col.replace("emission_totale_", "gain_pct_")] = (1 - totals[total_col] / totals["emission_totale_kg_co2e"]) * 100
totals.head()

In [ ]:
dim_route.to_csv(gold_dir / "dim_route.csv", index=False)
dim_stop.to_csv(gold_dir / "dim_stop.csv", index=False)
dim_vehicle.to_csv(gold_dir / "dim_vehicle.csv", index=False)
fact_route_leg.to_csv(gold_dir / "fact_route_leg.csv", index=False)
totals.to_csv(gold_dir / "kpi_route_totals.csv", index=False)
print(f"Gold sauvegardé dans {gold_dir}")